In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/X_train_pre.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/images_analysis.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/lang.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/translation.json
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/df_train_clean.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/label_mapping.json
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/df_train.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/df_val.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/X_test_pre.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/df_test.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/df_test_clean.csv


KeyboardInterrupt: 

In [4]:
import os, pickle
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (CamembertTokenizer, CamembertForSequenceClassification,
                          get_linear_schedule_with_warmup)
from torch.optim import AdamW
from sklearn.metrics import f1_score
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Vérification GPU ─────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu  = torch.cuda.device_count()
print(f"Device : {device}")
print(f"GPU(s) : {n_gpu}")
for i in range(n_gpu):
    print(f"   GPU {i} : {torch.cuda.get_device_name(i)}")

# ── Chemins ──────────────────────────────────────────────────────
INPUT_DIR = Path('/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed')
WORK_DIR  = Path('/kaggle/working')

random_state = 42
torch.manual_seed(random_state)
np.random.seed(random_state)

Device : cuda
GPU(s) : 2
   GPU 0 : Tesla T4
   GPU 1 : Tesla T4


In [5]:
df_train = pd.read_csv(INPUT_DIR / 'df_train.csv', index_col=0)
df_val   = pd.read_csv(INPUT_DIR / 'df_val.csv',   index_col=0)
df_test  = pd.read_csv(INPUT_DIR / 'df_test.csv',  index_col=0)

print(f"Train : {df_train.shape}")
print(f"Val   : {df_val.shape}")
print(f"Test  : {df_test.shape}")
print(f"Colonnes : {df_train.columns.tolist()}")

# Colonne texte — pour CamemBERT on préfère text_fr (non lemmatisé)
# BERT gère mieux le texte original que le texte lemmatisé
TEXT_COL = 'text_fr' if 'text_fr' in df_train.columns \
           else 'text_lemmatise' if 'text_lemmatise' in df_train.columns \
           else 'text'
print(f"\n✅ Colonne texte : '{TEXT_COL}'")

# Encodage labels → 0 à 26
classes      = sorted(df_train['prdtypecode'].unique())
class2idx    = {c: i for i, c in enumerate(classes)}
idx2class    = {i: c for i, c in enumerate(classes)}
NUM_CLASSES  = len(classes)
print(f"   Classes : {NUM_CLASSES}")

df_train['label'] = df_train['prdtypecode'].map(class2idx)
df_val['label']   = df_val['prdtypecode'].map(class2idx)
df_test['label']  = df_test['prdtypecode'].map(class2idx)

Train : (109566, 8)
Val   : (8492, 8)
Test  : (8492, 8)
Colonnes : ['productid', 'imageid', 'text_fr', 'lang', 'categorie', 'imagefile', 'text_lemmatise', 'prdtypecode']

✅ Colonne texte : 'text_fr'
   Classes : 27


In [6]:
MAX_LENGTH = 128   # 128 suffit pour les désignations + descriptions courtes

tokenizer = CamembertTokenizer.from_pretrained('camembert-base')
print(f"✅ Tokenizer chargé")
print(f"   Exemple : {tokenizer.tokenize(df_train[TEXT_COL].iloc[0][:50])}")

class RakutenDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Création des datasets
train_dataset = RakutenDataset(
    df_train[TEXT_COL].fillna('').values,
    df_train['label'].values, tokenizer, MAX_LENGTH)
val_dataset   = RakutenDataset(
    df_val[TEXT_COL].fillna('').values,
    df_val['label'].values,   tokenizer, MAX_LENGTH)
test_dataset  = RakutenDataset(
    df_test[TEXT_COL].fillna('').values,
    df_test['label'].values,  tokenizer, MAX_LENGTH)

# DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)

print(f"\n✅ Datasets créés")
print(f"   Train batches : {len(train_loader)}")
print(f"   Val batches   : {len(val_loader)}")
print(f"   Test batches  : {len(test_loader)}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer chargé
   Exemple : ['▁Je', 'sus', '▁-', '▁Cahiers', '▁Du', '▁Libre', '▁', 'Avenir', '▁-', '▁Pr', 'être', '▁autrement']

✅ Datasets créés
   Train batches : 3424
   Val batches   : 266
   Test batches  : 266


In [7]:
model = CamembertForSequenceClassification.from_pretrained(
    'camembert-base',
    num_labels=NUM_CLASSES
)

# Multi-GPU si disponible
if n_gpu > 1:
    model = nn.DataParallel(model)
    print(f"✅ DataParallel activé sur {n_gpu} GPUs")

model = model.to(device)

# Paramètres
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Modèle chargé")
print(f"   Paramètres totaux     : {total_params:,}")
print(f"   Paramètres entraînés  : {trainable_params:,}")

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ DataParallel activé sur 2 GPUs
✅ Modèle chargé
   Paramètres totaux     : 110,642,715
   Paramètres entraînés  : 110,642,715


In [8]:
EPOCHS       = 3
LR           = 2e-5
WARMUP_RATIO = 0.1

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"✅ Optimiseur configuré")
print(f"   Epochs         : {EPOCHS}")
print(f"   Total steps    : {total_steps:,}")
print(f"   Warmup steps   : {warmup_steps:,}")
print(f"   Learning rate  : {LR}")
print(f"\n⏱️  Temps estimé : ~1h à 1h30 avec T4 x2")

✅ Optimiseur configuré
   Epochs         : 3
   Total steps    : 10,272
   Warmup steps   : 1,027
   Learning rate  : 2e-05

⏱️  Temps estimé : ~1h à 1h30 avec T4 x2


In [9]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch['attention_mask'].to(device)
            labels    = batch['label'].to(device)

            outputs = model(input_ids=input_ids,
                            attention_mask=attn_mask,
                            labels=labels)
            total_loss += outputs.loss.mean().item()

            probs  = torch.softmax(outputs.logits, dim=1)
            preds  = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    f1  = f1_score(all_labels, all_preds, average='weighted')
    avg_loss = total_loss / len(loader)
    return f1, avg_loss, np.array(all_probs), np.array(all_preds)


# ── Entraînement ─────────────────────────────────────────────────
history     = []
best_f1_val = 0
best_epoch  = 0

print("🚀 Début de l'entraînement...\n")

for epoch in range(EPOCHS):
    model.train()
    total_loss  = 0
    all_preds   = []
    all_labels  = []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        labels    = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids,
                        attention_mask=attn_mask,
                        labels=labels)
        loss = outputs.loss.mean()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        loop.set_postfix(loss=f"{loss.item():.4f}")

    # Métriques epoch
    train_f1   = f1_score(all_labels, all_preds, average='weighted')
    avg_loss   = total_loss / len(train_loader)
    val_f1, val_loss, _, _ = evaluate(model, val_loader, device)

    history.append({
        'epoch': epoch+1, 'train_loss': avg_loss,
        'train_f1': train_f1, 'val_f1': val_f1
    })

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"   Train Loss : {avg_loss:.4f}  |  Train F1 : {train_f1:.4f}")
    print(f"   Val   F1   : {val_f1:.4f}")

    # Sauvegarde meilleur modèle
    if val_f1 > best_f1_val:
        best_f1_val = val_f1
        best_epoch  = epoch + 1
        model_to_save = model.module if hasattr(model, 'module') else model
        model_to_save.save_pretrained(WORK_DIR / 'camembert_best')
        tokenizer.save_pretrained(WORK_DIR / 'camembert_best')
        print(f"   💾 Meilleur modèle sauvegardé (F1={val_f1:.4f})")

print(f"\n✅ Entraînement terminé")
print(f"   Meilleur epoch : {best_epoch}  |  Meilleur F1 Val : {best_f1_val:.4f}")

🚀 Début de l'entraînement...



Epoch 1/3: 100%|██████████| 3424/3424 [25:29<00:00,  2.24it/s, loss=0.6438]



Epoch 1/3
   Train Loss : 1.4442  |  Train F1 : 0.7035
   Val   F1   : 0.8489


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   💾 Meilleur modèle sauvegardé (F1=0.8489)


...

   💾 Meilleur modèle sauvegardé (F1=0.8751)


Epoch 3/3: 100%|██████████| 3424/3424 [25:47<00:00,  2.21it/s, loss=0.0490]



Epoch 3/3
   Train Loss : 0.2474  |  Train F1 : 0.9310
   Val   F1   : 0.8826


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   💾 Meilleur modèle sauvegardé (F1=0.8826)

✅ Entraînement terminé
   Meilleur epoch : 3  |  Meilleur F1 Val : 0.8826


In [10]:
# Chargement meilleur modèle
from transformers import CamembertForSequenceClassification
best_model = CamembertForSequenceClassification.from_pretrained(
    WORK_DIR / 'camembert_best')
if n_gpu > 1:
    best_model = nn.DataParallel(best_model)
best_model = best_model.to(device)

# Évaluation
val_f1,  val_loss,  val_probs,  val_preds  = evaluate(best_model, val_loader,  device)
test_f1, test_loss, test_probs, test_preds = evaluate(best_model, test_loader, device)

print(f"✅ Résultats finaux CamemBERT")
print(f"   F1 Val  (weighted) : {val_f1:.4f}")
print(f"   F1 Test (weighted) : {test_f1:.4f}")
print(f"\n📊 Comparaison :")
print(f"   TF-IDF + SVC  → {0.8121:.4f}")
print(f"   CamemBERT     → {test_f1:.4f}  (+{test_f1-0.8121:.4f})")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Résultats finaux CamemBERT
   F1 Val  (weighted) : 0.8826
   F1 Test (weighted) : 0.8777

📊 Comparaison :
   TF-IDF + SVC  → 0.8121
   CamemBERT     → 0.8777  (+0.0656)


In [11]:
# Ces fichiers seront utilisés pour la fusion avec VGG16
np.save(WORK_DIR / 'camembert_val_probs.npy',  val_probs)
np.save(WORK_DIR / 'camembert_test_probs.npy', test_probs)

# Sauvegarde du mapping classes
with open(WORK_DIR / 'class_mapping.pkl', 'wb') as f:
    pickle.dump({'class2idx': class2idx, 'idx2class': idx2class,
                 'classes': classes}, f)

# Résumé training
pd.DataFrame(history).to_csv(WORK_DIR / 'camembert_training_history.csv',
                              index=False)

print("✅ Fichiers sauvegardés dans /kaggle/working/ :")
for f in sorted(WORK_DIR.iterdir()):
    print(f"   {f.name:<45} {f.stat().st_size/1024/1024:.2f} MB")

✅ Fichiers sauvegardés dans /kaggle/working/ :
   .virtual_documents                            0.00 MB
   camembert_best                                0.00 MB
   camembert_test_probs.npy                      0.87 MB
   camembert_training_history.csv                0.00 MB
   camembert_val_probs.npy                       0.87 MB
   class_mapping.pkl                             0.00 MB
